# 📦 Day 03a: Encapsulation — How Data Gets Wrapped Layer-by-Layer

---

## 🎯 What You'll Master Today
- Data encapsulation and decapsulation
- Headers at each layer
- MTU, fragmentation, reassembly
- Packet structure visualization

---

### 🎭 The Analogy

Encapsulation is like putting a letter in envelopes:
1. You write a letter (**Application data**)
2. Put it in an envelope with TO/FROM address (**Transport: port numbers**)
3. Put that in a shipping box with city addresses (**Network: IP addresses**)
4. Stick a shipping label on the box (**Data Link: MAC addresses**)
5. Load it on a truck (**Physical: bits on wire**)

```
╔══════════════════════════════════════════════════════════════════╗
║                    ENCAPSULATION                                ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  Application:  [        DATA         ]                          ║
║                                                                  ║
║  Transport:    [TCP HDR][    DATA     ]  ← Segment              ║
║                                                                  ║
║  Network:      [IP HDR][TCP HDR][DATA]   ← Packet               ║
║                                                                  ║
║  Data Link:    [ETH HDR][IP][TCP][DATA][CRC]  ← Frame           ║
║                                                                  ║
║  Physical:     101010110101001110...      ← Bits                 ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Encapsulation Simulator
# ============================================

class EthernetFrame:
    HEADER_SIZE = 14   # 6B dst_mac + 6B src_mac + 2B type
    TRAILER_SIZE = 4   # 4B CRC

class IPPacket:
    HEADER_SIZE = 20   # 20 bytes (no options)

class TCPSegment:
    HEADER_SIZE = 20   # 20 bytes (no options)

def simulate_encapsulation(app_data: str):
    data_bytes = len(app_data.encode())
    
    print(f"  ═══ Encapsulation of '{app_data}' ═══\n")
    
    # Application
    http_header = "GET / HTTP/1.1\r\nHost: example.com\r\n\r\n"
    app_size = len(http_header.encode()) + data_bytes
    print(f"  Layer 7 (Application):")
    print(f"    HTTP header ({len(http_header.encode())}B) + data ({data_bytes}B) = {app_size}B\n")
    
    # Transport
    segment_size = TCPSegment.HEADER_SIZE + app_size
    print(f"  Layer 4 (Transport):")
    print(f"    TCP header ({TCPSegment.HEADER_SIZE}B) + app data ({app_size}B) = {segment_size}B")
    print(f"    [SrcPort|DstPort|SeqNum|AckNum|Flags|Window|Checksum|...][Payload]\n")
    
    # Network
    packet_size = IPPacket.HEADER_SIZE + segment_size
    print(f"  Layer 3 (Network):")
    print(f"    IP header ({IPPacket.HEADER_SIZE}B) + segment ({segment_size}B) = {packet_size}B")
    print(f"    [Version|IHL|TOS|TotalLen|ID|Flags|TTL|Proto|Checksum|SrcIP|DstIP][Payload]\n")
    
    # Data Link
    frame_size = EthernetFrame.HEADER_SIZE + packet_size + EthernetFrame.TRAILER_SIZE
    print(f"  Layer 2 (Data Link):")
    print(f"    Eth header ({EthernetFrame.HEADER_SIZE}B) + packet ({packet_size}B) + CRC ({EthernetFrame.TRAILER_SIZE}B) = {frame_size}B")
    print(f"    [DstMAC|SrcMAC|EtherType][Payload][CRC]\n")
    
    # Physical
    print(f"  Layer 1 (Physical):")
    print(f"    {frame_size * 8} bits transmitted on wire + preamble (8B)")
    print(f"    Total on wire: {(frame_size + 8) * 8} bits\n")
    
    # Summary
    overhead = frame_size - data_bytes
    print(f"  📊 Summary:")
    print(f"    User data:       {data_bytes} bytes")
    print(f"    Total on wire:   {frame_size} bytes")
    print(f"    Protocol overhead: {overhead} bytes ({overhead/frame_size*100:.1f}%)")

simulate_encapsulation("Hello, World!")

In [ ]:
# ============================================
# 2. Ethernet Frame Structure
# ============================================

print("""
  ╔══════════════════════════════════════════════════════════════════╗
  ║              ETHERNET FRAME (IEEE 802.3)                        ║
  ╠══════════════════════════════════════════════════════════════════╣
  ║                                                                  ║
  ║  ┌──────────┬──────────┬──────────┬───────────────┬──────────┐  ║
  ║  │ Preamble │ Dst MAC  │ Src MAC  │  EtherType    │ Payload  │  ║
  ║  │  8 bytes │ 6 bytes  │ 6 bytes  │   2 bytes     │ 46-1500  │  ║
  ║  └──────────┴──────────┴──────────┴───────────────┴──────────┘  ║
  ║                                                   ┌──────────┐  ║
  ║                                                   │   CRC    │  ║
  ║                                                   │  4 bytes │  ║
  ║                                                   └──────────┘  ║
  ║                                                                  ║
  ║  EtherType values:                                              ║
  ║    0x0800 = IPv4                                                 ║
  ║    0x0806 = ARP                                                  ║
  ║    0x86DD = IPv6                                                 ║
  ║                                                                  ║
  ║  MTU (Max Transmission Unit) = 1500 bytes (Ethernet payload)    ║
  ║  Min frame = 64 bytes (pad if needed)                           ║
  ║  Max frame = 1518 bytes (without preamble)                      ║
  ╚══════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# ============================================
# 3. IP Fragmentation Simulator
# ============================================

def fragment_packet(data_size, mtu=1500):
    """Simulate IP fragmentation."""
    ip_header = 20  # bytes
    max_payload = mtu - ip_header  # 1480 bytes
    # Payload must be multiple of 8 (fragment offset granularity)
    max_payload = (max_payload // 8) * 8  # 1480 (already aligned)
    
    fragments = []
    offset = 0
    remaining = data_size
    frag_id = 0
    
    while remaining > 0:
        payload = min(remaining, max_payload)
        if remaining > max_payload:
            # Not last fragment — must be multiple of 8
            payload = max_payload
            more_fragments = True
        else:
            more_fragments = False
        
        fragments.append({
            "id": frag_id,
            "offset": offset,
            "payload_size": payload,
            "total_size": payload + ip_header,
            "MF": more_fragments,
        })
        offset += payload
        remaining -= payload
        frag_id += 1
    
    return fragments

# Fragment a 4000-byte payload through MTU=1500
print("  IP Fragmentation: 4000B data through MTU=1500\n")
frags = fragment_packet(4000, mtu=1500)

print(f"  Original: 4000B data + 20B IP header = 4020B")
print(f"  MTU: 1500B → max payload per fragment: 1480B")
print(f"  Fragments needed: {len(frags)}\n")

for f in frags:
    mf = "MF=1 (more)" if f['MF'] else "MF=0 (last)"
    print(f"    Fragment {f['id']}: offset={f['offset']:4}  payload={f['payload_size']:4}B  total={f['total_size']:4}B  {mf}")

print("\n  ⚠️  Fragmentation is BAD for performance:")
print("    - Receiver must buffer and reassemble")
print("    - If ONE fragment lost → entire datagram retransmitted")
print("    - Solution: Path MTU Discovery (PMTUD) — avoid fragmentation!")

In [ ]:
# ============================================
# 4. Packet Trace — Full Journey
# ============================================

print("""
  ═══ PACKET JOURNEY: Browser → Web Server ═══

  You type: https://example.com

  STEP 1: DNS Resolution (get IP)
    Browser → DNS query → 93.184.216.34

  STEP 2: TCP 3-Way Handshake
    Client → SYN           → Server
    Client ← SYN-ACK       ← Server
    Client → ACK           → Server

  STEP 3: TLS Handshake (HTTPS)
    Client → ClientHello   → Server
    Client ← ServerHello   ← Server (cert, key exchange)
    Client → Finished      → Server

  STEP 4: HTTP Request (encapsulated)
  ┌──────────────────────────────────────────────────────────┐
  │ Ethernet Header                                          │
  │  ┌──────────────────────────────────────────────────┐    │
  │  │ IP Header (src=192.168.1.10, dst=93.184.216.34) │    │
  │  │  ┌────────────────────────────────────────────┐  │    │
  │  │  │ TCP Header (src=54321, dst=443)            │  │    │
  │  │  │  ┌──────────────────────────────────────┐  │  │    │
  │  │  │  │ TLS Record                           │  │  │    │
  │  │  │  │  ┌────────────────────────────────┐  │  │  │    │
  │  │  │  │  │ HTTP: GET / HTTP/1.1           │  │  │  │    │
  │  │  │  │  └────────────────────────────────┘  │  │  │    │
  │  │  │  └──────────────────────────────────────┘  │  │    │
  │  │  └────────────────────────────────────────────┘  │    │
  │  └──────────────────────────────────────────────────┘    │
  │ CRC                                                      │
  └──────────────────────────────────────────────────────────┘

  STEP 5: Routing
    Frame → Switch (L2, MAC) → Router (L3, IP) → ISP → Internet
    Note: MAC changes at each hop, IP stays the same!

  STEP 6: Server receives, decapsulates, processes, responds
""")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | What is encapsulation? | Each layer adds its header to data from layer above. Reverse = decapsulation |
| 2 | What is MTU? | Maximum Transmission Unit = 1500 bytes for Ethernet. Larger → fragmentation |
| 3 | What changes at each router hop? | MAC addresses change (new L2 header). IP addresses stay the same (unless NAT) |
| 4 | Why is fragmentation bad? | One lost fragment = entire datagram lost. Use Path MTU Discovery instead |
| 5 | Name PDU at each layer? | Application=Data, Transport=Segment, Network=Packet, Data Link=Frame, Physical=Bits |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Encapsulation: each layer wraps data + header         │
## │  • Ethernet MTU = 1500B, min frame = 64B                 │
## │  • IP fragments if packet > MTU (avoid with PMTUD!)     │
## │  • MAC changes per hop, IP stays constant                │
## │  • Overhead: headers add ~54+ bytes per packet           │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **HTTP & HTTPS** — The protocol powering the web